# Creating a basic RAG pipeline with Haystack - 01

In this notebook, we will walk through the foundational steps of building a basic Retrieval-Augmented Generation (RAG) pipeline using Haystack.

We'll begin by exploring Haystack's Document data class and learn how to convert raw files into structured Document objects. Next, we'll inspect the content and metadata of these documents, analyze token counts per chunk, and understand the importance of consistent tokenization with our embedding model. 

We’ll then set up an indexing pipeline to embed and store documents in a Haystack Document Store. Finally, we’ll construct a RAG pipeline that includes a prompt template, a query embedder, a retriever, and an LLM. 

In this notebook, embeddings will be generated using Torch inference, while the LLM will run via vLLM. In the next notebook, we’ll serve both models through vLLM for a more production-ready setup.

## Setup Environment

### Import packages

In [ ]:
from pathlib import Path
from haystack import Pipeline, Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.components.converters import DOCXToDocument
from haystack.utils import Secret
from haystack import Document
import textwrap
import time
from dotenv import dotenv_values
from transformers import AutoTokenizer
from typing import List

#### Helper Functions

In [ ]:
def calc_tokens(document:str, sample_text_length:int = 500) -> tuple[int, List[str]]:
    """Calculates the token count of a given text

    Args:
        document (str): The document we want to process.
        sample_text_length (int, optional): How much of that text we want to process. Use -1 to calculate the tokens of the whole text. Defaults to 500.

    Returns:
        tuple[int, List[str]]: First variable is the token count and the second is the actual tokens.
    """

    sample_text = document if sample_text_length == -1 else document[:sample_text_length]

    tokens = tokenizer.tokenize(sample_text)
    token_count = len(tokens)

    return [token_count, tokens]

def convert_documents(documents_dir:Path) -> List[Document]:
    """Return a list of Haystack Documents from a local path that contains our files

    Args:
        documents_dir (Path): The path that contains our files.

    Returns:
        List[Document]: A list that contains the converted Haystack Documents
    """

    FILES = [file.resolve() for file in documents_dir.rglob("*") if file.is_file()]
    converter = DOCXToDocument()

    docs = []
    for file in FILES:
        result = converter.run(sources=[file])
        docs.extend(result["documents"])  # Append the converted documents

    return docs

#### Setup VLLM Connection Parameters

In [ ]:
llm_config = dotenv_values('/nvme/scratch/edu29/llm.env')

### Convert data to Haystack Documents

Haystack uses these abstraction called *Documents*. They can hold text, tables, and binary data.

They have the following unique features:

- Unique ID for each document.
- Multiple content types are supported.
- Custom metadata and scoring for advanced document management.
- Optional embeddings for AI-based applications

**Example:**

```python
@dataclass
class Document(metaclass=_BackwardCompatible):
    id: str = field(default="")
    content: Optional[str] = field(default=None)
    blob: Optional[ByteStream] = field(default=None)
    meta: Dict[str, Any] = field(default_factory=dict)
    score: Optional[float] = field(default=None)
    embedding: Optional[List[float]] = field(default=None)
    sparse_embedding: Optional[SparseEmbedding] = field(default=None)
```

---

To convert our files do Haystack Documents, we need to use a *converter*. In our case, since we only have .docx documents, we can go ahead and use Haystack's DOCXToDocument converter.

In [ ]:
# Convert our files into Haystack Documents
DOCUMENTS_DIR = Path("./dummy_data/documents_dir")
docs = convert_documents(documents_dir=DOCUMENTS_DIR)

# Initialize a document store
document_store = InMemoryDocumentStore()

# Define the Document Embedder and Document Splitter
doc_embedder = SentenceTransformersDocumentEmbedder(model="BAAI/bge-base-en-v1.5")
splitter = DocumentSplitter(split_by="word", split_length=15)

#### Inspect a sample document from our database

In [ ]:
print(f"""
Type: {type(docs[0])}
ID: {docs[0].id}
""")

#### Content

In [ ]:
print(docs[0].content)

#### Metadata

In [ ]:
docs[0].meta

## Indexing Documents and performing RAG

### Setup Indexing components

To be able to use our documents we need to perform 4 things:

1. Convert our files to Haystack Documents
2. Split each document into chunks
3. Turn each chunk into embeddings with an *embedder*.
4. Store them in a Haystack *Document Store* so they can be accessed later on.

### Why do we split our documents into document chunks?

There are various reasons to split the whole document into smaller chunks. Some of them include:

1. Our models may have a limited context size where fitting the whole document can exceed that limit, throwing an error.
2. I will give two examples:
   1. Imagine we have a 35 pages long document, and the information we want to retrieve is on page 5 of that document. Why feed the LLM 35 pages of useless information, filling its memory, instead of feeding it 1 page?
   2. Imagine that we ask the LLM something, and it needs to access 5 different documents to answer it. Again, it is wasteful to load all the 5 documents to the LLM instead of just the 5 different parts of the document.
   3. As a by-product, giving the information useless information increases the chance of hallucinations.

#### Inspecting token size of a document

Run the code cell below to see how many tokens a document is.

**Note:** We will calculate the tokens for the first 500 characters, feel free to insert -1 to calculate the tokens for the whole document.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

token_count, tokens = calc_tokens(document=docs[0].content, sample_text_length=500) # Set sample_text_length to -1 to calculate the whole document
print(f'Token Count: {token_count}')
print(f"Tokens: {tokens}")

You can observe above the following:

1. The tokens for the whole document is more than 512 tokens. If we were using a smaller embedding model, such as [BAAI/bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5) which has a max context-length of 512, then we would run into errors.
2. Complex words get split:

    - Annual --> Annu, al </br>
    - hackathon --> Hack, a, thon</br>
    - Company --> Comp, any</br>
    - transformative --> transforma, tive</br>
    - innovate --> in, nova, te

---



Each tokenizer tokenizes words differently. When you count tokens, make sure to use the tokenizer your embedding model uses.

Run the following test:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

token_count, tokens = calc_tokens(document=docs[0].content, sample_text_length=500) # Set sample_text_length to -1 to calculate the whole document
print(f'Token Count: {token_count}')
print(f"Tokens: {tokens}")

You can observe 3 things:

1. Token count between each tokenizer is different.
2. Tokens are calculated differently. They are split at different points, and the denomenator use different as well. This tokenizer uses **#** and the other is using **__**
3. We get the following error:

Token indices sequence length is longer than the specified maximum sequence length for this model (1072 > 512). Running this sequence through the model will result in indexing errors

---

The main takeaway from this, is to try to always use the tokenizer that was used for the embedding model that you choose.

### Extract embeddings and store to Document Store

All models (embedding or not) have a limit on how much data they can process. We've seen this above when we were counting the tokens of each document.

This is one of the reasons on why we need to split our documents into smaller chunks.

#### More reasons to split our documents:

- Another reason is to avoid giving our LLM useless information.

Lets say we have a document with 35 pages, and the relevant information that answers our query is found on a small segment of page 5 of our document. Why not pass only that segment to the LLM instead of the whole document that is filled with irrelevant information?

Splitting the document ensures our model receives only relevant information, and it makes sure that our context size won't exceed our LLMs limit.

- Adding on the point above:

Imagine that we have a database with a lot of documents. The user asks a query about a product, and the information about that product is split into multiple documents. If we do not split our documents, we won't be able to pass into the LLM multiple documents before reaching our context-length limit.

If we were to split all the documents into smaller chunks, we would be able to pass multiple chunks to the model from various documents.

In [ ]:
split_docs = splitter.run(documents=docs) # Split the documents into chunks

start_time = time.time()
doc_embedder.warm_up()

docs_with_embeddings = doc_embedder.run(split_docs['documents']) # Calculate the embeddings for each chunk

end_time = time.time()
print(f'Inference time: {end_time - start_time:.2f} seconds')

document_store.write_documents(docs_with_embeddings["documents"]) # Write the docments into the document store.
print(f"Stored {len(docs_with_embeddings['documents'])} documents with embeddings in the document store.")

We have a simple timer in the above code cell to compare it on the next notebook where we will run the embedder through VLLM.

---

### Initialize RAG

#### Prompt Template for user

This prompt template will be used by our LLM to generate a response based on our Query.

Specifically, the LLM will read this text from top to bottom:

- It will read the task, which is to respond to the user's query using the **provided context**.
- It will then read some **General Guidelines**.
- Then it will read the **provided context**.
- And finally it will read the **user's query**.

You can see that we pass the **context** and **user's query** through this template. This is why we use a prompt builder later on. This component constructs prompts dynamically by processing chat messages.

Specifically, the *ChatPromptBuilder* component creates prompts using static or dynamic templates written in Jinja2 syntax, by processing a list of chat messages. The templates contain placeholders like {{ variable }} that are filled with values provided during runtime. You can use it for static prompts set at initialization or change the templates and variables dynamically while running.

In [ ]:
with open("/nvme/scratch/edu29/dummy_data/RAG_TEMPLATE.txt") as rag_file:
    rag_template = rag_file.read()

print(rag_template)

user_prompt_template = ChatMessage.from_user(rag_template)

#### RAG Components

For the actual RAG pipeline we have the following components:

- The *text_embedder* which takes the user's query and turns it into embeddings
- The *retriever* which retrieves the relevant documents
- The *chat_generator* which is our LLM
- The *promt_builder* which was explained above.

In [ ]:
# Note: For the Text Embedder, make sure you are using the same embedding model you used during the indexing phase.
text_embedder = SentenceTransformersTextEmbedder(model="BAAI/bge-base-en-v1.5")
retriever = InMemoryEmbeddingRetriever(document_store)
chat_generator = OpenAIChatGenerator(api_key=Secret.from_token(llm_config['VLLM_API_KEY']),
                                     api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
                                     model=llm_config['MODEL_NAME'],
)
                                                                       
prompt_builder = ChatPromptBuilder(variables=["query", "documents"], required_variables=["query", "documents"])

# Initialize RAG pipeline
basic_rag_pipeline = Pipeline()

# Add the RAG Components
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", chat_generator)

# Connect the input/output of each component
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

We connect each component by defining the inputs and outputs.

For example:

- The *text_embedder* will take the user's query as an input and output *embeddings*. These embeddings will be the input of the *retriever*. The *retriever* will take that input as *query_embedding* and output a list of *documents* that are similar to that query.
- Then, the *retriever* outputs the documents we mentioned, and pass them into our *prompt_builder*.
- Finally, the *prompt_builder* will output the prompt and send it as an input to the *llm*.

#### Perform RAG on our data

Feel free to change the question to something else.

Our documents contain information about the following topics:

- Annual Hackathon the company is organizing
- Cybersecurity Awareness Month
- Employee Recognition Program
- New Office Layout Plan
- Office layout redesign plan
- Product X Launch Timeline
- Product Y Launch Timeline
- QuantumStream product CLI Usage
- QuantumStream product Data Encryption feature
- QuantumStream product Plugin System
- QuantumStream product REST API documentation
- QuantumStream product Scheduler feature
- QuantumStream product Scheduling tasks

---

Feel free to ask anything relating to these topics.

**Suggested prompts:**

- "Whats the purpose of the new office layout? Are we loosing our desks??"
- "I am a new employee at the company. Onboard me about the QuantumStream product."
- "I cannot find the relevant email about the Hackathon, can you tell me more details about it?"

In [ ]:
question = "I am a new employee at the company. Onboard me about the QuantumStream product." # Feel free to change this question

response = basic_rag_pipeline.run(
    data={
        "text_embedder": {"text": question}, 
        "prompt_builder": {"template": [user_prompt_template], "query": question}}
        )
formatted_text = response["llm"]["replies"][0].text
wrapped_text = "\n".join(
    textwrap.fill(line, width=120, subsequent_indent="  ") if line.strip() else line
    for line in formatted_text.splitlines()
)

print(wrapped_text)